In [ ]:
import glob
import os
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
from collections import defaultdict
from sklearn.model_selection import train_test_split



In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_path = "/content/drive/MyDrive/dev_phase/subtask1/train/"
files = glob.glob(os.path.join(base_path, "*.csv"))

data = {}

for file in files:
    lang = os.path.splitext(os.path.basename(file))[0]  # amh, arb, eng

    df = pd.read_csv(file)

    data[lang] = {
        "X": df["text"].tolist(),
        "y": df["polarization"].tolist(),
        "df": df
    }

print("Loaded languages:", sorted(data.keys()))


In [ ]:
model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
embedding_model = AutoModel.from_pretrained(model_name).to(device)
embedding_model.eval()

for p in embedding_model.parameters():
    p.requires_grad = False

def mean_pooling(model_output, attention_mask):
    token_embeds = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()
    sum_embeddings = torch.sum(token_embeds * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask


In [ ]:
def get_all_embeddings(texts, model, tokenizer, device, batch_size=32):
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model(**enc)
            embeddings = mean_pooling(outputs, enc['attention_mask'])
        all_embs.append(embeddings.cpu())
    return torch.cat(all_embs, dim=0)


embeddings_by_lang = {}

for lang, content in data.items():   # data[lang] has "X" and "y"
    print(f"Embedding language: {lang}")

    X_text = content["X"]
    y_labels = content["y"]

    X_emb = get_all_embeddings(X_text, embedding_model, tokenizer, device)
    y_tensor = torch.tensor(y_labels, dtype=torch.long)

    embeddings_by_lang[lang] = {
        "X": X_emb,
        "y": y_tensor
    }


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):
        # x: (B, 1, D)
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # last timestep
        return self.fc(out)


In [ ]:
def train_lstm_cv_stats(X, y, device, num_classes,
                        k=5, epochs=10, batch_size=32, lr=1e-3):

    X = X.unsqueeze(1)  # (N,1,D)

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    accs = []
    f1s  = []

    for fold, (tr, te) in enumerate(skf.split(X, y)):
        print(f"\nFold {fold+1}")

        X_tr, X_te = X[tr], X[te]
        y_tr, y_te = y[tr], y[te]

        train_ds = torch.utils.data.TensorDataset(X_tr, y_tr)
        test_ds  = torch.utils.data.TensorDataset(X_te, y_te)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        test_loader  = DataLoader(test_ds, batch_size=batch_size)

        model = LSTMClassifier(
            input_dim=X.shape[-1],
            hidden_dim=256,
            num_classes=num_classes
        ).to(device)

        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()

        # ---- train ----
        for ep in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)

                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()

        # ---- eval ----
        model.eval()
        preds = []

        with torch.no_grad():
            for xb, _ in test_loader:
                xb = xb.to(device)
                out = model(xb)
                preds.extend(out.argmax(1).cpu().numpy())

        acc = accuracy_score(y_te, preds)
        f1  = f1_score(y_te, preds, average="macro")

        print(f"ACC={acc:.4f} | F1={f1:.4f}")

        accs.append(acc)
        f1s.append(f1)

    return {
        "mean_acc": np.mean(accs),
        "std_acc":  np.std(accs),
        "mean_f1":  np.mean(f1s),
        "std_f1":   np.std(f1s)
    }


In [ ]:
results = []

for lang in embeddings_by_lang:

    print(f"\n######## {lang.upper()} ########")

    X = embeddings_by_lang[lang]["X"]
    y = embeddings_by_lang[lang]["y"]

    num_classes = len(torch.unique(y))

    stats = train_lstm_cv_stats(
        X, y,
        device=device,
        num_classes=num_classes,
        k=5,
        epochs=10,
        batch_size=32
    )

    results.append({
        "language": lang,
        "mean_acc": stats["mean_acc"],
        "std_acc": stats["std_acc"],
        "mean_macro_f1": stats["mean_f1"],
        "std_macro_f1": stats["std_f1"]
    })

final_df = pd.DataFrame(results)
final_df = final_df.sort_values("mean_macro_f1", ascending=False)

final_df


In [ ]:
final_df.to_csv("e5_lstm.csv", index=False)
